# Structured Output with Pydantic & LangChain

This notebook demonstrates extracting type-safe, validated structured data from LLMs using LangChain and Pydantic:
- **Agent Initialization**: Configuring `ChatOllama` and compiling an agent runtime with tools.
- **Basic Structured Output**: Defining flat schema models with Pydantic `BaseModel` and `Field`.
- **Enforcing Output Schemas**: Using `model.with_structured_output()` to guarantee structured parsing.
- **Nested & Relational Schemas**: Defining complex hierarchical data structures with nested Pydantic models and list types.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()  # reads OLLAMA_MODEL from .env (set per machine)
import langchain_ollama
from langchain_ollama import ChatOllama

from langchain.agents import create_agent

def get_weather(city:str)->str:
    """get weather from city"""
    return f"weather in {city} is sunny"

llm = ChatOllama(model=os.getenv("OLLAMA_MODEL", "qwen3:8b"), temperature=0, reasoning=True)

agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="You are a helpful assistant. Use tools when needed.",
)

agent

### 1. Agent Initialization & Tool Setup

- **Model Setup**: Initializes `ChatOllama` with the model named by `OLLAMA_MODEL` in `.env`, setting `temperature=0` for deterministic outputs and `reasoning=True`.
- **Tool Schema**: Implements `get_weather` with type hints and docstring definitions.
- **Agent Compilation**: Assembles the agent graph via `create_agent` with tool bindings and system instructions.

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(..., description="Name of the movie")
    main_actor: str = Field(..., description="Name of the main actor")
    famous_character: str = Field(..., description="name of the most famous character")
    director: str = Field(..., description="name of the director")
    rating: float = Field(..., description="rating of the movie")

model_with_structured_output = llm.with_structured_output(Movie)
resp = model_with_structured_output.invoke("give me details about Avengers end game")
resp

### 2. Basic Structured Output with Pydantic

- **Schema Definition**: Uses Pydantic's `BaseModel` and `Field(..., description=...)` to specify attributes, types, and descriptive constraints.
- **Structured Output Binding**: Calls `llm.with_structured_output(Movie)` to transform the model from returning unstructured text to returning validated `Movie` instances.
- **Type-Safe Extraction**: Invoking the model parses movie details (title, main actor, famous character, director, rating) directly into a typed Python object.

In [ ]:
from pydantic import BaseModel, Field

class Actors(BaseModel):
    name: str = Field(..., description="name of the actor")
    role: str = Field(..., description="name of the character")

class Movie(BaseModel):
    title: str = Field(..., description="Name of the movie")
    main_actor: str = Field(..., description="Name of the main actor")
    actors : list[Actors]
    genere : list[str]
    famous_character: str = Field(..., description="name of the most famous character")
    director: str = Field(..., description="name of the director")
    rating: float = Field(..., description="rating of the movie")

model_with_structured_output = llm.with_structured_output(Movie)
resp = model_with_structured_output.invoke("give me details about Oppenheimer")
resp

### 3. Nested Schemas & Complex Data Extraction

- **Hierarchical Modeling**: Defines an inner `Actors` schema (`name`, `role`) and nests it inside the parent `Movie` schema as `actors: list[Actors]`.
- **Collection Fields**: Supports string lists such as `genere: list[str]` to capture multi-value attributes.
- **Deep Extraction**: Extracts complete cast lists, genres, and metadata in a single structured invocation with full type validation.